# Official CODI boundary-aware selector confirmation

## Goal

Test whether forcing the first and last valid teacher trace tokens and using R-KV for four interior targets transfers more low-rank KV signal than unchanged R-KV, uniform, or random selection. The 5,000-example sample is explicitly disjoint from the completed selector-specificity experiment.

## 1. Choose the run scope

Run the audit first, followed by the complete collection and analysis. The prior seed-1 collection manifest is a required scientific input because it defines the exclusion set. Collection state is saved atomically to Drive and can be resumed by rerunning the same cell.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the pushed immutable commit before the final run.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"

RUN_AUDIT = True
RUN_COLLECTION = True
RUN_ANALYSIS = True

EXAMPLES = 5000
DATA_SEED = 2
RANDOM_SELECTOR_SEEDS = [101, 211, 307, 401]
BATCH_SIZE = 16
SHUFFLE_REPEATS = 4
SAVE_EVERY = 1000
PRECISION = "bfloat16"

GATE_RANK = 4
SELECTOR_SIGNAL_MARGIN = 0.01
SELECTOR_WIN_FRACTION = 0.60

## 2. Mount Drive and install the pinned environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import datetime
import json
import os
import pathlib
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("Pin RUN_COMMIT before the final collection to:", commit)

## 3. Verify GPU, code contracts, accuracy gate, and prior collection

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Colab GPU runtime"
gpu_name = torch.cuda.get_device_name(0)
print("Torch:", torch.__version__)
print("GPU:", gpu_name)
if "A100" not in gpu_name:
    print("Warning: this workflow runs on the current GPU, but the time estimate assumed an A100.")

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_official_codi.py",
        "tests/test_official_codi_kv.py",
        "tests/test_kv_compress.py",
        "tests/test_kv_cross_subspace.py",
        "tests/test_kv_reduced_rank.py",
        "tests/test_kv_selector_specificity.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

official_root = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_gpt2"
gate_candidates = sorted(official_root.glob("eval/revision_*/full_gsm8k/summary.json"))
passed = []
for candidate in gate_candidates:
    payload = json.loads(candidate.read_text())
    gate = payload.get("accuracy_gate", payload.get("gate"))
    status = gate if isinstance(gate, str) else (gate or {}).get("status")
    if status == "passed":
        passed.append(candidate)
assert passed, "Run colab_official_codi_validation.ipynb until full GSM8K passes"
REPRODUCTION_SUMMARY = passed[-1]

PRIOR_MANIFEST = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_selector_specificity" / "n5000_seed1" / "collection_manifest.json"
assert PRIOR_MANIFEST.is_file(), "The completed seed-1 selector manifest is missing"
prior = json.loads(PRIOR_MANIFEST.read_text())
assert prior["state"] == "complete"
assert prior["processed_examples"] == 5000
assert len(prior["sample_indices"]) == 5000
assert len(set(prior["sample_indices"])) == 5000
print("Passed reproduction summary:", REPRODUCTION_SUMMARY)
print("Prior exclusion manifest:", PRIOR_MANIFEST)
print("Prior sample fingerprint:", prior["indices_sha256"])

## 4. Define Drive-persistent execution

In [ ]:
OUTPUT_DIR = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_boundary_selector" / "n5000_seed2"
REPORT_ROOT = pathlib.Path(DRIVE_ROOT) / "reports" / "official_codi_boundary_selector"
LOG_ROOT = pathlib.Path(DRIVE_ROOT) / "logs" / "official_codi_boundary_selector"
for path in (OUTPUT_DIR, REPORT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(cmd, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, cmd)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, cmd))} ===\n")
        process = subprocess.Popen(
            list(map(str, cmd)), cwd=REPO_DIR, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code

def collection_command(output_dir, examples, audit_only=False):
    cmd = [
        sys.executable, "-u", "scripts/collect_official_codi_selector_subspaces.py",
        "--config", "configs/official_codi_gpt2.yaml",
        "--reproduction-summary", str(REPRODUCTION_SUMMARY),
        "--output-dir", str(output_dir),
        "--examples", str(examples),
        "--batch-size", str(BATCH_SIZE),
        "--shuffle-repeats", str(SHUFFLE_REPEATS),
        "--save-every", str(SAVE_EVERY),
        "--precision", PRECISION,
        "--device", "cuda",
        "--seed", str(DATA_SEED),
        "--random-selector-seeds", ",".join(map(str, RANDOM_SELECTOR_SEEDS)),
        "--include-boundary-rkv",
        "--exclude-manifest", str(PRIOR_MANIFEST),
    ]
    if audit_only:
        cmd.append("--audit-only")
    return cmd

## 5. Audit boundary retention and sample exclusion

The audit checks the seven selector arms, finite tensors, forced first and last indices for representative examples, and the scientific exclusion metadata without allocating the full moment collection.

In [ ]:
from IPython.display import JSON, Markdown, display

AUDIT_DIR = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_boundary_selector" / "audit_seed2"
if RUN_AUDIT:
    run_persisted(collection_command(AUDIT_DIR, BATCH_SIZE, audit_only=True), "boundary_selector_audit.log")
    audit = json.loads((AUDIT_DIR / "selection_audit.json").read_text())
    audit_manifest = json.loads((AUDIT_DIR / "collection_manifest.json").read_text())
    expected_selectors = {"boundary_rkv", "rkv", "uniform", *[f"random_seed{s}" for s in RANDOM_SELECTOR_SEEDS]}
    assert set(audit["selectors"]) == expected_selectors
    assert audit["student_latent_shape"][1:] == [12, 12, 6, 64]
    assert audit_manifest["sample_overlap_with_exclusion"] == 0
    assert audit_manifest["excluded_indices_count"] == 5000
    hybrid_rows = audit["selectors"]["boundary_rkv"]["representative_indices_layer0_head0"]
    trace_lengths = audit["teacher_trace_tokens"]["values_first_eight"]
    for row, count in zip(hybrid_rows, trace_lengths):
        if count > 0:
            assert 0 in row
            assert count - 1 in row
    for selector, values in audit["selectors"].items():
        assert values["selected_valid_fraction"] > 0.0
        assert values["finite_teacher_keys"] and values["finite_teacher_values"]
    display(JSON(audit))
else:
    print("Audit skipped")

## 6. Collect the complete disjoint 5,000-example statistics

All seven selector arms are accumulated from one teacher/student forward pass per batch. Rerun this cell after a disconnect to resume from the latest atomic Drive checkpoint.

In [ ]:
if RUN_COLLECTION:
    run_persisted(collection_command(OUTPUT_DIR, EXAMPLES), "collect_n5000_seed2.log")
    manifest = json.loads((OUTPUT_DIR / "collection_manifest.json").read_text())
    assert manifest["state"] == "complete"
    assert manifest["processed_examples"] == EXAMPLES
    assert manifest["include_boundary_rkv"]
    assert manifest["sample_overlap_with_exclusion"] == 0
    assert manifest["excluded_indices_count"] == 5000
    assert not set(manifest["sample_indices"]) & set(prior["sample_indices"])
    print(json.dumps({k: manifest[k] for k in ("state", "processed_examples", "selectors", "indices_sha256", "excluded_indices_sha256", "sample_overlap_with_exclusion")}, indent=2))
else:
    print("Collection skipped")

## 7. Run the preregistered candidate gate

Boundary-aware R-KV must pass its own actual-versus-shuffle gate and beat unchanged R-KV, uniform, and the per-group random median by at least 0.01 signal R² across at least 60 percent of matched groups.

In [ ]:
REPORT_PATH = REPORT_ROOT / "official_codi_n5000_seed2_boundary_selector.json"
if RUN_ANALYSIS:
    run_persisted(
        [
            sys.executable, "scripts/analyze_kv_boundary_selector.py",
            "--statistics", str(OUTPUT_DIR),
            "--output", str(REPORT_PATH),
            "--gate-rank", str(GATE_RANK),
            "--selector-signal-margin", str(SELECTOR_SIGNAL_MARGIN),
            "--selector-win-fraction", str(SELECTOR_WIN_FRACTION),
        ],
        "analyze_n5000_seed2.log",
    )
    display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
else:
    print("Analysis skipped")

## 8. Checks and next decision

In [ ]:
print("\nDurable collection")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name, f"{path.stat().st_size / (1024 ** 2):.1f} MiB")

print("\nDurable reports")
for path in sorted(REPORT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPORT_ROOT), f"{path.stat().st_size / 1024:.1f} KiB")

if REPORT_PATH.is_file():
    result = json.loads(REPORT_PATH.read_text())
    print("\nGate:", result["gate"]["status"])

display(Markdown("""
### Interpretation boundary

A positive gate supports boundary-aware R-KV as a stronger source of transferable linear KV signal than unchanged R-KV, uniform, and random selection on fresh disjoint data. It still does not establish answer causality or an accuracy improvement. Only after a positive gate should a compute-matched downstream distillation experiment begin. A negative gate means the exploratory position crossover did not generalize into a globally stronger selector.
"""))